In [0]:
/*
  Script: Load Gold
  Description:
    This script processes the tables in the silver table to create and load tables in the gold layer.
*/

SELECT '=================================================';
SELECT 'Loading dim_customers view';
SELECT '=================================================';

DROP VIEW IF EXISTS gold.dim_customers;

CREATE VIEW gold.dim_customers AS
(
  SELECT
    ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,
    cst_id AS customer_id,
    cst_key AS customer_number,
    cst_firstname AS first_name,
    cst_lastname AS last_name,
    cntry AS country,
    cst_marital_status AS marital_status,
    CASE
      WHEN gen = 'n/a' AND cst_gndr != 'n/a' THEN cst_gndr
      WHEN cst_gndr = 'n/a' AND gen != 'n/a' THEN gen
      ELSE cst_gndr
      END AS gender,
    bdate AS birth_date,
    cst_create_date AS create_date
  FROM silver.crm_cust_info t1
  LEFT JOIN silver.erp_cust_az12 t2
    ON t1.cst_key = t2.cid
  LEFT JOIN silver.erp_loc_a101 t3
    ON t1.cst_key = t3.cid
)
;

SELECT '=================================================';
SELECT 'Loading dim_products view';
SELECT '=================================================';

DROP VIEW IF EXISTS gold.dim_products;

-- Retain only the newest version of product info
CREATE VIEW gold.dim_products AS
(
  SELECT
    ROW_NUMBER() OVER (ORDER BY prd_start_dt, prd_key) AS product_key,
    prd_id AS product_id,
    prd_key AS product_number,
    prd_nm AS product_name,
    cat_id AS category_id,
    cat AS category_name,
    subcat AS subcategory_name,
    maintenance AS maintenance,
    prd_cost AS product_cost, 
    prd_line AS product_line,
    prd_start_dt AS start_date
  FROM
  (
    SELECT
    *
    FROM silver.crm_prd_info t1
    LEFT JOIN silver.erp_px_cat_g1v2 t2
      ON t1.prd_key = t2.id
    WHERE prd_end_dt IS NULL
  )
)
;

SELECT '=================================================';
SELECT 'Loading fact_sales view';
SELECT '=================================================';

DROP VIEW IF EXISTS gold.fact_sales;

CREATE VIEW gold.fact_sales AS
(
  SELECT
    sls_ord_num AS order_number,
    product_key,
    customer_key,
    sls_order_dt AS order_date,
    sls_ship_dt AS ship_date,
    sls_due_dt AS due_date,
    sls_sales AS sales_amount,
    sls_quantity AS quantity,
    sls_price AS price
  FROM silver.crm_sales_details t1
  LEFT JOIN gold.dim_products t2
    ON t1.sls_prd_key = t2.product_number
  LEFT JOIN gold.dim_customers t3
    ON t1.sls_cust_id = t3.customer_id
)
;
